# RAG-Based Profile Matching Evaluation & Benchmark

This notebook evaluates the retrieval accuracy and latency of different local embedding models under multiple filtering settings, and compares them to public benchmarks of paid providers (OpenAI, Cohere).

In [1]:
import sys
import time
import numpy as np
import chromadb
from pathlib import Path

# Add src/ to path
sys.path.append(str(Path("..") / "src"))

import config
from resume_rag import ResumeRAGPipeline
from job_matcher import JobMatcher
from fs_tools import list_files, read_file
from eval_config import GROUND_TRUTH, BENCHMARK_DATA

## Step 1: Metric Helper Functions

We define standard Information Retrieval (IR) metrics to evaluate the search performance:
- **Precision@K**: Proportion of retrieved items in top-K that are relevant.
- **Recall@K**: Proportion of all relevant items retrieved in top-K.
- **Mean Average Precision (MAP)**: Measures rank-aware precision across all relevant docs.
- **Mean Reciprocal Rank (MRR)**: Evaluates the rank of the first relevant result.

In [2]:
def calculate_precision_recall_at_k(retrieved, ground_truth, k):
    top_k = retrieved[:k]
    hits = [candidate for candidate in top_k if candidate in ground_truth]
    precision = len(hits) / k
    recall = len(hits) / len(ground_truth) if len(ground_truth) > 0 else 0.0
    return precision, recall

def calculate_ap(retrieved, ground_truth):
    if not ground_truth:
        return 0.0
    hits = 0
    sum_precisions = 0.0
    for i, candidate in enumerate(retrieved):
        if candidate in ground_truth:
            hits += 1
            precision_at_i = hits / (i + 1)
            sum_precisions += precision_at_i
    return sum_precisions / len(ground_truth)

def calculate_rr(retrieved, ground_truth):
    for i, candidate in enumerate(retrieved):
        if candidate in ground_truth:
            return 1.0 / (i + 1)
    return 0.0

## Step 2: Live Local Models Evaluation

We evaluate the following three local HuggingFace/SentenceTransformers models:
1. `sentence-transformers/all-MiniLM-L6-v2` (Current default)
2. `BAAI/bge-small-en-v1.5` (High-performance retrieval model)
3. `sentence-transformers/paraphrase-MiniLM-L3-v2` (Extremely fast, lower parameter count)

We evaluate them in two modes:
- **Filtered Mode**: With metadata constraints (experience & must-have skills).
- **Unfiltered Mode**: Raw retrieval ranking (semantic + keyword scoring only).

In [3]:
models_to_evaluate = [
    ("sentence-transformers/all-MiniLM-L6-v2", "resumes_all_minilm"),
    ("BAAI/bge-small-en-v1.5", "resumes_bge_small"),
    ("sentence-transformers/paraphrase-MiniLM-L3-v2", "resumes_paraphrase_minilm")
]

jds = list_files(config.JOB_DESCRIPTIONS_DIR)
jds_data = []
for jd_file in sorted(jds, key=lambda f: f['name']):
    data = read_file(jd_file['path'])
    if data['success']:
        jds_data.append((jd_file['name'], data['content']))

results_comparison = {}

for model_name, col_name in models_to_evaluate:
    # Clean collection for evaluation
    client = chromadb.PersistentClient(path=config.VECTOR_DB_PATH)
    try:
        client.delete_collection(col_name)
    except Exception:
        pass

    t0 = time.time()
    pipeline = ResumeRAGPipeline(model_name=model_name, collection_name=col_name)
    pipeline.ingest_directory(config.RESUMES_DIR)
    ingest_time = time.time() - t0

    matcher = JobMatcher(model_name=model_name, collection_name=col_name)

    model_metrics = {
        "filtered": {"p1": [], "p3": [], "p5": [], "r3": [], "r5": [], "map": [], "mrr": [], "latency": []},
        "unfiltered": {"p1": [], "p3": [], "p5": [], "r3": [], "r5": [], "map": [], "mrr": [], "latency": []}
    }

    for jd_name, jd_content in jds_data:
        ground_truth = set(GROUND_TRUTH.get(jd_name, []))
        if not ground_truth:
            continue

        for mode in ["filtered", "unfiltered"]:
            apply_filters = (mode == "filtered")
            t_start = time.time()
            matches_res = matcher.match(jd_content, k=10, apply_filters=apply_filters)
            t_latency = (time.time() - t_start) * 1000

            retrieved_candidates = [m['candidate_name'] for m in matches_res.get('top_matches', [])]

            p1, _ = calculate_precision_recall_at_k(retrieved_candidates, ground_truth, k=1)
            p3, r3 = calculate_precision_recall_at_k(retrieved_candidates, ground_truth, k=3)
            p5, r5 = calculate_precision_recall_at_k(retrieved_candidates, ground_truth, k=5)
            ap = calculate_ap(retrieved_candidates, ground_truth)
            rr = calculate_rr(retrieved_candidates, ground_truth)

            model_metrics[mode]["p1"].append(p1)
            model_metrics[mode]["p3"].append(p3)
            model_metrics[mode]["p5"].append(p5)
            model_metrics[mode]["r3"].append(r3)
            model_metrics[mode]["r5"].append(r5)
            model_metrics[mode]["map"].append(ap)
            model_metrics[mode]["mrr"].append(rr)
            model_metrics[mode]["latency"].append(t_latency)

    results_comparison[model_name] = {
        "ingest_time_s": ingest_time,
        "filtered": {metric: np.mean(vals) for metric, vals in model_metrics["filtered"].items()},
        "unfiltered": {metric: np.mean(vals) for metric, vals in model_metrics["unfiltered"].items()}
    }

## Step 3: Comparison and Analysis

Let's display the local models evaluation table:

In [4]:
headers = ["Model", "Mode", "Ingest (s)", "Avg Latency (ms)", "P@1", "P@3", "R@3", "P@5", "R@5", "MAP", "MRR"]
row_fmt = "{:<45} | {:<10} | {:<10} | {:<16} | {:<5} | {:<5} | {:<5} | {:<5} | {:<5} | {:<5} | {:<5}"
print("-" * 135)
print(row_fmt.format(*headers))
print("-" * 135)
for model_name, info in results_comparison.items():
    short_name = model_name.split("/")[-1]
    for mode in ["filtered", "unfiltered"]:
        m = info[mode]
        print(row_fmt.format(
            short_name,
            mode,
            f"{info['ingest_time_s']:.2f}",
            f"{m['latency']:.2f}",
            f"{m['p1']:.2f}",
            f"{m['p3']:.2f}",
            f"{m['r3']:.2f}",
            f"{m['p5']:.2f}",
            f"{m['r5']:.2f}",
            f"{m['map']:.2f}",
            f"{m['mrr']:.2f}"
        ))
print("-" * 135)

---------------------------------------------------------------------------------------------------------------------------------------
Model                                         | Mode       | Ingest (s) | Avg Latency (ms) | P@1   | P@3   | R@3   | P@5   | R@5   | MAP   | MRR  
---------------------------------------------------------------------------------------------------------------------------------------
all-MiniLM-L6-v2                              | filtered   | 7.95       | 30.51            | 0.60  | 0.53  | 0.45  | 0.36  | 0.49  | 0.52  | 0.60 
all-MiniLM-L6-v2                              | unfiltered | 7.95       | 12.93            | 0.80  | 0.60  | 0.65  | 0.40  | 0.69  | 0.75  | 0.83 
bge-small-en-v1.5                             | filtered   | 40.37      | 19.53            | 0.60  | 0.53  | 0.45  | 0.40  | 0.56  | 0.55  | 0.60 
bge-small-en-v1.5                             | unfiltered | 40.37      | 15.77            | 0.80  | 0.53  | 0.59  | 0.48  | 0.96  | 0.79  |

### Observation & Insights
- **Metadata Filtering Impact**: Filtered mode limits recall. For example, some candidates have 5 years of experience when the JD asks for 6+ years. Strict filtering removes them, reducing the recall. In unfiltered mode, the models achieve **Recall@5 up to 0.96** (BGE small) because the candidates are allowed to be ranked purely on semantic and keyword overlap, allowing matching systems to evaluate close/soft fits.
- **BGE Small vs. MiniLM**: BAAI's `bge-small-en-v1.5` achieves the highest MAP (0.79) and Recall@5 (0.96) in unfiltered mode, confirming its superior retrieval capability over `all-MiniLM-L6-v2` (MAP: 0.75). However, MiniLM is much lighter to download and faster to ingest.

## Step 4: Comparison with Proprietary / API-based Providers

Since API-based models (OpenAI, Cohere) require payment and API keys, we compare them based on public MTEB (Massive Text Embedding Benchmark) Retrieval scores and cost parameters:

In [5]:
benchmark_headers = ["Model Name", "Provider", "Dimension", "Cost per 1M Tokens", "MTEB Retrieval Avg", "Deployment", "License"]
benchmark_fmt = "{:<25} | {:<30} | {:<9} | {:<19} | {:<18} | {:<11} | {:<11}"
print("-" * 140)
print(benchmark_fmt.format(*benchmark_headers))
print("-" * 140)
for b in BENCHMARK_DATA:
    print(benchmark_fmt.format(
        b["Model Name"],
        b["Provider"],
        b["Dimension"],
        b["Cost per 1M Tokens"],
        b["MTEB Retrieval Avg"],
        b["Deployment Type"],
        b["License"]
    ))
print("-" * 140)

--------------------------------------------------------------------------------------------------------------------------------------------
Model Name                | Provider                       | Dimension | Cost per 1M Tokens  | MTEB Retrieval Avg | Deployment  | License    
--------------------------------------------------------------------------------------------------------------------------------------------
text-embedding-3-small    | OpenAI (Paid API)              | 1536      | $0.02               | 52.2               | API-based   | Proprietary
text-embedding-ada-002    | OpenAI (Paid API)              | 1536      | $0.10               | 49.3               | API-based   | Proprietary
embed-english-v3.0        | Cohere (Paid API)              | 1024      | $0.10               | 56.2               | API-based   | Proprietary
bge-small-en-v1.5         | BAAI (Local/Free)              | 384       | $0.00 (Local)       | 51.1               | Local       | MIT        
all-Mini

### Proprietary vs. Free Local Comparison Notes
1. **Performance**: Cohere `embed-english-v3.0` leads the MTEB benchmark at 56.2, closely followed by OpenAI `text-embedding-3-small` (52.2) and local `bge-small-en-v1.5` (51.1). Crucially, **our free local BGE model performs on par with OpenAI's latest text-embedding-3-small** while being completely free and private.
2. **Privacy & Control**: Paid API models require sending resume details to third-party endpoints. In HR tech, keeping candidate details locally hosted (using models like BGE and MiniLM) ensures data privacy and GDPR compliance.
3. **Cost**: Running local models incurs a cost of $0.00 per token, making scaling to millions of resumes extremely cost-effective compared to paid endpoints.